# RE_204 · Formigó: memoritzar no és aprendre

**Bloc 2 · Supervisat · repte 4 de 6 · durada: 2 sessions de 2 h**

> **Repte resolt.** Una manera de resoldre'l, no l'única. Els números del text surten d'una execució real del quadern.

Treballes al laboratori d'una planta de formigó. Per saber si una barreja aguanta, es fa una proveta
i es trenca a la premsa quan té 28 dies (o 7, o 90...). Això vol dir **esperar setmanes**. El cap de
laboratori et pregunta si es pot **predir la resistència** (en MPa) a partir de la recepta i l'edat.

Aquest repte és de **regressió** (es prediu un número, no una classe), i va de dues trampes: **l'escala
de les columnes** i **el model que s'aprèn les dades de memòria**. La pregunta: **quin error real
pots prometre al laboratori, i com saps que no t'estàs enganyant?**

L'enunciat complet, amb el diccionari de columnes, és a [RE_204](../RE_204_formigo.ipynb). Si encara no l'has fet, fes-lo abans de llegir això.

## Què necessites saber abans

- k veïns i el parany de les escales: [ML_01 §3, §4 i §6](../../01_teoria/ML_01_knn.ipynb) i [EX_01 exercicis 3-4](../../02_practica/EX_01_wine.ipynb)
- Fins on fer créixer un arbre: [ML_02 §6](../../01_teoria/ML_02_arbres.ipynb) i [EX_04 exercici 4](../../02_practica/EX_04_fronteres.ipynb)
- `StandardScaler` i el parany de `fit_transform` contra `transform`: [EX_00 §2](../../02_practica/EX_00_caixa_eines.ipynb)
- `cross_val_score` i `GridSearchCV`: [EX_00 §9](../../02_practica/EX_00_caixa_eines.ipynb)
- La regressió és nova: els models es diuen igual però acaben en `Regressor`, i en lloc de matriu de confusió es mesura **quant s'equivoca de mitjana**.

## Les dades

**Concrete Compressive Strength** (Yeh, 1998; UCI), a OpenML amb `data_id=44959`. Llicència CC BY
4.0. Cada fila és **una proveta de formigó**: quants kg de cada component hi ha per m³ de barreja
(ciment, escòria, cendres volants, aigua, superplastificant, grava, sorra), l'**edat** en dies quan es
va trencar, i la **resistència a compressió** en MPa (`strength`), que és l'objectiu.

Documentació: <https://archive.ics.uci.edu/dataset/165/concrete+compressive+strength>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

df = fetch_openml(data_id=44959, as_frame=True).frame
print(df.shape)
df.head()

## Nivell 1 · Explorar

### 1.1 · Unitats i rangs

Mira el mínim, el màxim, la mitjana i la desviació de cada columna. Quina té el rang més gran, i quina el més petit? Si calculessis una distància euclidiana entre dues provetes sense tocar res, quina columna hi manaria?

In [ ]:
assert df.shape == (1030, 9) and df.isna().sum().sum() == 0
print(df.describe().T[["min", "max", "mean", "std"]].round(1))
print((df.max() - df.min()).sort_values())

> **Resultat.** Rangs molt diferents: `fine_aggregate` i `cement` es mouen en centenars de kg, `superplasticizer` en desenes. Sense escalar, la distància la decideixen les columnes de rang gran.

### 1.2 · L'edat i els duplicats

Quants valors diferents té `age`? Què vol dir per a un model que sigui una columna amb pocs valors? Quantes files duplicades hi ha? I quantes provetes tenen **la mateixa recepta i edat** però una resistència diferent? Què implica això per a l'error mínim que pot tenir qualsevol model?

In [ ]:
print(df["age"].value_counts().sort_index())
dup = df.duplicated().sum()
dup_x = df.duplicated(subset=df.columns.drop("strength")).sum()
assert dup_x >= dup
print("Duplicats exactes:", dup, "| mateixa recepta i edat:", dup_x)

> **Resultat.** `age` té 14 valors (dies de curat típics: 3, 7, 28, 56, 90...), i el 28 és el més freqüent (425 provetes). Hi ha 25 duplicats exactes i 34 files que repeteixen recepta i edat: les que tenen resistència diferent són soroll de laboratori que cap model pot encertar.

### 1.3 · Què va amb la resistència

Calcula la correlació de cada columna amb `strength` i fes el gràfic de dispersió de les dues amb més correlació (en valor absolut). La relació sembla una recta?

In [ ]:
corr = df.corr()["strength"]
assert np.isclose(corr["strength"], 1) and corr.between(-1, 1).all()
print(corr.drop("strength").sort_values().round(3))
top2 = corr.drop("strength").abs().sort_values().index[-2:]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for a, c in zip(ax, top2):
    a.scatter(df[c], df["strength"], s=6, alpha=0.5); a.set_xlabel(c); a.set_ylabel("strength")
plt.show()

> **Resultat.** `cement` és la més correlacionada (0,50), després `superplasticizer` (0,37) i `age` (0,33); `water` és la més negativa (−0,29). L'edat no fa una recta: la resistència creix de pressa els primers dies i després s'estanca. El gràfic és la clau per entendre per què un arbre pot anar millor que una recta.

## Nivell 2 · Respondre

### 2.1 · kNN amb i sense escalar

Parteix en entrenament i test (25 %, `random_state=42`). Entrena un `KNeighborsRegressor` amb `k=5` **sense escalar** i un altre dins d'un `Pipeline` amb `StandardScaler`. Quin és l'error absolut mitjà (MAE) de test de cadascun? Quina diferència hi ha, en MPa?

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, r2_score

X, y = df.drop(columns="strength"), df["strength"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

knn_cru = KNeighborsRegressor(n_neighbors=5).fit(X_train, y_train)
pipe = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)).fit(X_train, y_train)
assert np.allclose(pipe[0].transform(X_train).mean(axis=0), 0)
print("mitjanes del test escalat:", pipe[0].transform(X_test).mean(axis=0).round(2))

mae_cru = mean_absolute_error(y_test, knn_cru.predict(X_test))
mae_esc = mean_absolute_error(y_test, pipe.predict(X_test))
print(f"MAE sense escalar: {mae_cru:.2f} MPa | escalat: {mae_esc:.2f} MPa")

> **Resultat.** Sorpresa: aquí escalar gairebé no canvia res (6,96 MPa sense escalar, 6,85 escalat). És una troballa en si mateixa: el parany de les escales no sempre és espectacular, i el 3.1 n'explica el perquè. Trobar-ho i dir-ho, sense forçar la conclusió esperada, també és part de la feina.

### 2.2 · La teoria, al codi: una predicció de kNN a mà

Agafa la primera proveta de test. Amb l'escalador del pipeline, calcula tu la distància euclidiana d'aquesta proveta a **totes** les d'entrenament, queda't amb les 5 més properes i fes la mitjana de la seva resistència. Compara-ho amb `pipe.predict`. Comprova també que els teus 5 veïns són els que diu `kneighbors`.

In [ ]:
A = pipe[0].transform(X_train)
x0 = pipe[0].transform(X_test.iloc[[0]])
dist = np.sqrt(((A - x0) ** 2).sum(axis=1))
idx = np.argsort(dist)[:5]
pred_ma = y_train.iloc[idx].mean()
pred_sk = pipe.predict(X_test.iloc[[0]])[0]
print(f"a mà: {pred_ma:.3f} | scikit-learn: {pred_sk:.3f} | real: {y_test.iloc[0]:.3f}")
assert np.isclose(pred_ma, pred_sk)
assert set(idx) == set(pipe[-1].kneighbors(x0)[1][0])

> **Resultat.** Coincideix. Si algú obté un número diferent, gairebé sempre és perquè ha calculat les distàncies sense escalar, o amb un escalador ajustat amb tot el dataset.

### 2.3 · La teoria, al codi: MAE i $R^2$

Implementa el MAE i el $R^2$ amb NumPy i compara'ls amb `mean_absolute_error` i `r2_score` per al kNN escalat. Què vol dir un $R^2$ de 0? I un de negatiu? Comprova-ho amb un «model» que prediu sempre la mitjana de l'entrenament.

In [ ]:
def mae(yv, p):
    return np.mean(np.abs(np.asarray(yv) - p))

def r2(yv, p):
    yv = np.asarray(yv)
    return 1 - np.sum((yv - p) ** 2) / np.sum((yv - yv.mean()) ** 2)

p = pipe.predict(X_test)
assert np.isclose(mae(y_test, p), mean_absolute_error(y_test, p))
assert np.isclose(r2(y_test, p), r2_score(y_test, p))
p_mitjana = np.full(len(y_test), y_train.mean())
print(f"kNN escalat: MAE {mae(y_test, p):.2f}, R2 {r2(y_test, p):.3f}")
print(f"sempre la mitjana: MAE {mae(y_test, p_mitjana):.2f}, R2 {r2(y_test, p_mitjana):.4f}")

> **Resultat.** El model de la mitjana té un $R^2$ de −0,0004 (gairebé 0) i un MAE de 13,32 MPa. El kNN escalat té $R^2$ 0,718 i MAE 6,85. Un $R^2$ negatiu vol dir pitjor que predir la mitjana.

### 2.4 · La corba del sobreajust

Entrena arbres de regressió amb `max_depth` de 1 a 20 i també sense límit (`None`). Dibuixa el MAE d'entrenament i el de test segons la profunditat. A partir de quina profunditat l'arbre comença a **memoritzar**? Com ho veus al gràfic?

In [ ]:
from sklearn.tree import DecisionTreeRegressor

prof = list(range(1, 21)) + [None]
tr, te = [], []
for d in prof:
    a = DecisionTreeRegressor(max_depth=d, random_state=42).fit(X_train, y_train)
    tr.append(mean_absolute_error(y_train, a.predict(X_train)))
    te.append(mean_absolute_error(y_test, a.predict(X_test)))
eix = list(range(1, 22))
plt.plot(eix, tr, "o-", label="entrenament"); plt.plot(eix, te, "s-", label="test")
plt.xticks(eix, [str(d) for d in prof], fontsize=7); plt.xlabel("max_depth"); plt.ylabel("MAE (MPa)")
plt.legend(); plt.show()
print(pd.DataFrame({"train": tr, "test": te}, index=[str(d) for d in prof]).round(2).T)
assert tr[-1] < 1

> **Resultat.** L'error d'entrenament baixa fins a 0,06 MPa sense límit; el de test baixa fins a ≈ 4,5 MPa cap a profunditat 16-17 i després es queda a 4,77. La corba de test és plana des de profunditat ~10, i el forat amb la d'entrenament (més de 4 MPa) no deixa de créixer. La distància entre les dues corbes és el sobreajust: el que el model «sap» de memòria i no serveix per a provetes noves.

### 2.5 · Triar sense mirar el test

Si tries la profunditat (o la `k`) mirant el test, el test ja no mesura res de nou. Fes servir **validació creuada de 5 plecs sobre l'entrenament** per triar la millor profunditat de l'arbre i la millor `k` del kNN escalat. Després avalua **una sola vegada** al test cada model triat. Quin error real pots prometre al laboratori?

In [ ]:
from sklearn.model_selection import cross_val_score

cv_arbre = {}
for d in [3, 5, 7, 9, 11, 13, 15, None]:
    s = cross_val_score(DecisionTreeRegressor(max_depth=d, random_state=42), X_train, y_train,
                        cv=5, scoring="neg_mean_absolute_error")
    assert len(s) == 5 and (s < 0).all()
    cv_arbre[str(d)] = -s.mean()
cv_knn = {}
for k in [1, 2, 3, 5, 7, 10, 15]:
    s = cross_val_score(make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k)), X_train, y_train,
                        cv=5, scoring="neg_mean_absolute_error")
    cv_knn[k] = -s.mean()
print("arbre (CV):", {k: round(v, 2) for k, v in cv_arbre.items()})
print("kNN   (CV):", {k: round(v, 2) for k, v in cv_knn.items()})

d_best = min(cv_arbre, key=cv_arbre.get); d_best = None if d_best == "None" else int(d_best)
k_best = min(cv_knn, key=cv_knn.get)
arbre = DecisionTreeRegressor(max_depth=d_best, random_state=42).fit(X_train, y_train)
knn = make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=k_best)).fit(X_train, y_train)
print(f"arbre max_depth={d_best}: test MAE {mean_absolute_error(y_test, arbre.predict(X_test)):.2f}")
print(f"kNN k={k_best}: test MAE {mean_absolute_error(y_test, knn.predict(X_test)):.2f}")

> **Resultat.** La validació creuada tria `max_depth=15` (MAE de CV 4,77) i `k=3` (6,92). Al test fan 4,42 i 6,81 MPa. El que es pot prometre és el MAE de test del model triat (≈ 4,4 MPa per a l'arbre), no el millor número de la corba de l'2.4, que es va triar mirant el test.

## Nivell 3 · Més enllà

### 3.1 · Quina columna fa mal sense escalar?

Al kNN sense escalar, quina columna és la «culpable»? Prova d'escalar **totes menys una** (cada vegada una de diferent) i mira quan l'error s'espatlla més. Coincideix amb el que esperaves de l'1.1?

In [ ]:
from sklearn.compose import ColumnTransformer

res = {}
for c in X.columns:
    altres = [x for x in X.columns if x != c]
    ct = ColumnTransformer([("esc", StandardScaler(), altres)], remainder="passthrough")
    m = make_pipeline(ct, KNeighborsRegressor(n_neighbors=5)).fit(X_train, y_train)
    res[c] = mean_absolute_error(y_test, m.predict(X_test))
print(pd.Series(res).sort_values(ascending=False).round(2))

> **Resultat.** Aquí hi ha la troballa bona: deixar sense escalar els àrids (`coarse_aggregate` 10,3, `fine_aggregate` 10,0) espatlla molt l'error, però deixar sense escalar **`age`** el **millora** (5,37, millor que tot escalat, 6,85). Una columna sense escalar i de rang gran mana a la distància: si és la més informativa (l'edat), és com donar-li més pes. Per això a l'2.1 l'escalat gairebé no guanyava: sense escalar, l'edat (rang 364) ja manava. Escalar iguala els pesos, no els optimitza.

### 3.2 · I una recta?

Entrena una `LinearRegression` (escalada) i compara-la amb els dos models triats a l'2.5. Mira'n els coeficients: quadren amb el que diu un enginyer (més ciment, més resistència; més aigua, menys)? Per què la recta perd contra l'arbre? Pensa en el gràfic de l'edat de l'1.3.

In [ ]:
from sklearn.linear_model import LinearRegression

lin = make_pipeline(StandardScaler(), LinearRegression()).fit(X_train, y_train)
print("recta MAE:", round(mean_absolute_error(y_test, lin.predict(X_test)), 2))
print(pd.Series(lin[-1].coef_, index=X.columns).sort_values().round(2))

Xl_tr, Xl_te = X_train.assign(log_age=np.log(X_train["age"])), X_test.assign(log_age=np.log(X_test["age"]))
lin2 = make_pipeline(StandardScaler(), LinearRegression()).fit(Xl_tr, y_train)
print("recta amb log(age) MAE:", round(mean_absolute_error(y_test, lin2.predict(Xl_te)), 2))

> **Resultat.** La recta queda clarament per darrere (MAE 7,99). Els signes tenen sentit físic (ciment +13,1, aigua −2,4). Afegir `log(age)` la baixa a 5,54: la no-linealitat era sobretot l'edat.

## Conclusió

Es pot prometre un error d'uns 4,4 MPa: és el MAE de test de l'arbre amb la profunditat triada
per validació creuada (15), mesurat una sola vegada al final. L'arbre sense límit s'equivoca 0,06 MPa
a l'entrenament i gairebé 5 al test: aquesta distància és memòria, no aprenentatge. El kNN queda per
darrere (6,8 MPa), i escalar-lo gairebé no canvia res perquè sense escalar ja manava l'edat, que és
la columna més útil (deixar-la sense escalar el baixa a 5,37). La recta fa 7,99 MPa perquè la
resistència no creix en línia recta amb l'edat; amb `log(age)` baixa a 5,54.